# Gesture Recognition — Training trên Google Colab

**Bước chuẩn bị (làm 1 lần):**
1. Zip thư mục `Do_an` thành `Do_an.zip`
2. Upload `Do_an.zip` lên Google Drive (thư mục gốc `MyDrive/`)
3. Vào `Runtime` → `Change runtime type` → chọn **T4 GPU** → Save
4. Chạy từng cell theo thứ tự

> Notebook tự giải nén vào `/content/Do_an` (SSD Colab) để train nhanh hơn.  
> Kết quả tự động copy về Drive sau khi xong.

## 1. Kiểm tra GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu}  |  VRAM: {mem:.1f} GB")
    DEVICE = "cuda"
else:
    print("Khong co GPU! Vao Runtime > Change runtime type > T4 GPU")
    DEVICE = "cpu"

print(f"PyTorch: {torch.__version__}  |  Device: {DEVICE}")

## 2. Mount Drive & Giải nén project

In [ ]:
import os, zipfile, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PROJECT_DIR  = Path('/content/Do_an')
ZIP_ON_DRIVE = DRIVE_ROOT / 'Do_an.zip'
DIR_ON_DRIVE = DRIVE_ROOT / 'Do_an'
STAMP_FILE   = PROJECT_DIR / '.extracted_at'

def _needs_extract():
    """Tra ve True neu can giai nen (lan dau hoac zip moi hon thu muc)."""
    if not PROJECT_DIR.exists() or not (PROJECT_DIR / 'config.yaml').exists():
        return True
    if ZIP_ON_DRIVE.exists() and STAMP_FILE.exists():
        if ZIP_ON_DRIVE.stat().st_mtime > STAMP_FILE.stat().st_mtime:
            print("ZIP moi hon thu muc hien tai — giai nen lai de cap nhat code...")
            return True
    return False

if not _needs_extract():
    print(f"Da co {PROJECT_DIR} va ZIP chua thay doi — dung lai.")

elif ZIP_ON_DRIVE.exists():
    print(f"Giai nen {ZIP_ON_DRIVE} ...")
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    with zipfile.ZipFile(ZIP_ON_DRIVE, 'r') as zf:
        zf.extractall('/content/')
    if not (PROJECT_DIR / 'config.yaml').exists():
        candidates = [p for p in Path('/content').iterdir()
                      if p.is_dir() and p.name not in ('drive', 'sample_data')]
        if candidates:
            candidates[0].rename(PROJECT_DIR)
    STAMP_FILE.touch()
    print(f"Giai nen xong: {PROJECT_DIR}")

elif DIR_ON_DRIVE.exists():
    print("Copy tu Drive vao /content/ ...")
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(DIR_ON_DRIVE, PROJECT_DIR)
    STAMP_FILE.touch()
    print(f"Copy xong: {PROJECT_DIR}")

else:
    raise FileNotFoundError(
        "Khong tim thay Do_an.zip hoac thu muc Do_an tren Drive.\n"
        "Hay upload Do_an.zip vao MyDrive/ truoc."
    )

os.chdir(PROJECT_DIR)
print(f"Working dir: {os.getcwd()}")
print(f"Files      : {[f for f in os.listdir() if not f.startswith('.')]}")

## 3. Cài đặt thư viện & package

In [ ]:
!pip install pyyaml tqdm -q
!pip install -e "/content/Do_an" -q

import sys
sys.path.insert(0, str(PROJECT_DIR))

from training import REPO_ROOT, load_config
cfg = load_config()
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"max_epochs  : {cfg['training']['max_epochs']}")
print(f"patience    : {cfg['training']['patience']}")

## 4. Tiền xử lý dữ liệu
> Bỏ qua nếu `processed_50hz/` đã có trong zip.

In [ ]:
from pathlib import Path
processed_dir = Path(cfg['paths']['processed'])
csv_files = list(processed_dir.rglob('*.csv')) if processed_dir.exists() else []

if csv_files:
    print(f"Da co {len(csv_files)} file processed — bo qua.")
else:
    print("Bat dau preprocess...")
    !python scripts/preprocess.py

## 5. Thống kê dataset

In [ ]:
from training.dataset import load_manifest, split_manifest

manifest = load_manifest(processed_dir)
splits   = split_manifest(
    manifest,
    test_subjects=cfg['split']['test_subjects'],
    val_subjects=cfg['split']['val_subjects'],
)
print(f"Tong  : {len(manifest)}")
print(f"Train : {len(splits['train'])}  Val: {len(splits['val'])}  Test: {len(splits['test'])}")
print(f"Classes  : {manifest['label_id'].nunique()}")
print(f"Subjects : {sorted(manifest['subject_id'].unique())}")

## 6. Train CNN1D

In [ ]:
!python scripts/train_dl.py --model cnn --device {DEVICE}

## 7. Train CNN + BiLSTM

In [ ]:
!python scripts/train_dl.py --model lstm --device {DEVICE}

## 8. Train Transformer

In [ ]:
!python scripts/train_dl.py --model transformer --device {DEVICE}

## 9. Train Late Fusion Random Forest

In [ ]:
!python scripts/train_rf_latefusion.py --fusion soft

## 10. Đánh giá tất cả model trên tập Test

In [ ]:
!python scripts/evaluate.py

## 11. Export model tốt nhất sang demo/

In [ ]:
!python scripts/export_best.py

## 12. Sinh TẤT CẢ figures cho báo cáo

Cell này tự động sinh toàn bộ:
- **EDA**: phân bố mẫu, độ dài chuỗi, tín hiệu từng cử chỉ
- **Loss / Accuracy curves**: mỗi DL model
- **Confusion Matrix**: từng model + so sánh cạnh nhau
- **F1-score per class**: bar chart so sánh các model
- **T-SNE**: không gian đặc trưng của từng DL model
- **Bảng hiệu năng**: accuracy, params, inference time

In [ ]:
import json
import torch
import numpy as np
import joblib
from pathlib import Path
from torch.utils.data import DataLoader

from training import load_config, REPO_ROOT
from training.dataset import (
    load_manifest, split_manifest, load_labels_json,
    IMUDataset, _read_csv_sequence, resample_1d,
)
from training.features import extract_features_demo
from training.models import CNN1D, LSTMModel, TransformerModel
from training.models_rf import LateFusionRF
from training.trainer import evaluate as eval_dl
from visualization.plot_tsne import extract_dl_features
from visualization.report_generator import ReportGenerator

cfg        = load_config()
gen        = ReportGenerator(cfg)
paths      = cfg['paths']
split_cfg  = cfg['split']
target_len = cfg['preprocessing']['target_len']

# ── Load manifest & label list ────────────────────────────────────────────
processed_dir = Path(paths['processed'])
manifest   = load_manifest(processed_dir)
splits     = split_manifest(
    manifest,
    test_subjects=split_cfg['test_subjects'],
    val_subjects=split_cfg['val_subjects'],
)
test_df   = splits['test']
label_ids = load_labels_json(Path(paths['labels_json']))
n_classes = len(label_ids)
print(f"Test samples: {len(test_df)}  |  Classes: {n_classes}")

# ── 1. EDA figures ────────────────────────────────────────────────────────
print("\n--- EDA ---")
gen.generate_eda()

# ── 2. DL models ─────────────────────────────────────────────────────────
DL_CONFIGS = {
    'cnn':         (CNN1D,            'results_cnn'),
    'lstm':        (LSTMModel,        'results_lstm'),
    'transformer': (TransformerModel, 'results_transformer'),
}

preds_by_model   = {}
performance_rows = []

for model_key, (ModelClass, results_key) in DL_CONFIGS.items():
    results_dir = Path(paths[results_key])
    if not results_dir.exists():
        print(f"  Skip {model_key}: chua co ket qua")
        continue
    runs = [p for p in results_dir.glob('run_*') if (p / 'meta.json').exists()]
    if not runs:
        print(f"  Skip {model_key}: khong co run nao")
        continue

    best_run = max(
        runs,
        key=lambda p: json.loads((p / 'meta.json').read_text()).get('val_accuracy', 0)
    )
    print(f"\n[{model_key.upper()}] {best_run.name}")

    # Loss & Accuracy curves
    hist_path = best_run / 'history.json'
    if hist_path.exists():
        gen.fig_loss_acc_from_file(model_key, hist_path)

    # Load normalization
    norm = json.loads((best_run / 'normalization.json').read_text())
    mean = np.array(norm['mean'], dtype=np.float32)
    std  = np.array(norm['std'],  dtype=np.float32)

    # Test DataLoader
    test_ds     = IMUDataset(test_df, processed_dir.parent, label_ids, target_len, mean, std)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

    # Load model
    model = ModelClass(input_channels=6, num_classes=n_classes)
    ckpt  = torch.load(best_run / 'best.pt', map_location='cpu', weights_only=True)
    model.load_state_dict(ckpt['model'])

    # Evaluate
    _, y_true, y_pred = eval_dl(model, test_loader, device=DEVICE)
    preds_by_model[model_key] = (y_true, y_pred)
    test_acc = float((y_true == y_pred).mean())
    print(f"  Test acc: {test_acc:.4f}")

    # Confusion matrix
    gen.fig_confusion_matrix(model_key, y_true, y_pred)

    # T-SNE
    feats, feat_labels = extract_dl_features(model, test_loader, device=DEVICE)
    if feats.ndim == 2 and feats.shape[0] > 0:
        gen.fig_tsne(model_key, feats, feat_labels)
    else:
        print(f"  T-SNE skip: khong lay duoc features")

    # Performance row
    meta = json.loads((best_run / 'meta.json').read_text())
    performance_rows.append({
        'model':    model_key,
        'val_acc':  meta['val_accuracy'],
        'top1_acc': test_acc,
        'top5_acc': test_acc,
        'params_m': sum(p.numel() for p in model.parameters()) / 1e6,
        'gflops':   0.0,
        'infer_ms': 0.0,
    })

# ── 3. RF model ───────────────────────────────────────────────────────────
rf_dir = Path(paths['results_rf'])
if rf_dir.exists():
    rf_runs = [p for p in rf_dir.glob('run_*') if (p / 'model.pkl').exists()]
    if rf_runs:
        best_rf = max(rf_runs, key=lambda p: p.stat().st_mtime)
        print(f"\n[RF] {best_rf.name}")

        rf_model = LateFusionRF.load(str(best_rf / 'model.pkl'))

        seqs, y_rf_true = [], []
        for _, row in test_df.iterrows():
            seq = _read_csv_sequence(row['path'], processed_dir.parent)
            if seq is None:
                continue
            seqs.append(resample_1d(seq, target_len))
            lbl = str(row['label_id'])
            y_rf_true.append(label_ids.index(lbl) if lbl in label_ids else 0)

        if seqs:
            X_rf      = extract_features_demo(np.stack(seqs))       # (N, 24)
            y_rf_arr  = np.array(y_rf_true, dtype=np.int64)
            y_rf_pred = rf_model.predict(X_rf)
            preds_by_model['rf_latefusion'] = (y_rf_arr, y_rf_pred)
            rf_acc = float((y_rf_arr == y_rf_pred).mean())
            print(f"  Test acc: {rf_acc:.4f}")

            gen.fig_confusion_matrix('rf_latefusion', y_rf_arr, y_rf_pred)

            rf_meta = json.loads((best_rf / 'meta.json').read_text()) \
                      if (best_rf / 'meta.json').exists() else {}
            performance_rows.append({
                'model':    'rf_latefusion',
                'val_acc':  rf_meta.get('val_accuracy', rf_acc),
                'top1_acc': rf_acc,
                'top5_acc': rf_acc,
                'params_m': 0.0,
                'gflops':   0.0,
                'infer_ms': 0.0,
            })

# ── 4. So sanh tat ca model ───────────────────────────────────────────────
if len(preds_by_model) >= 2:
    print("\n--- Comparison figures ---")
    gen.fig_confusion_comparison(preds_by_model)
    gen.fig_f1_comparison(preds_by_model)

if performance_rows:
    gen.fig_performance_table(performance_rows)

# ── Summary ───────────────────────────────────────────────────────────────
all_figs = sorted(gen.fig_dir.glob('*.png'))
print(f"\n=== Tong cong {len(all_figs)} figures → {gen.fig_dir} ===")
for f in all_figs:
    print(f"  {f.name}")

## 13. Kết quả tổng hợp

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

fig_dir = Path(cfg['paths']['figures'])

print("=== Test Accuracy ===")
for r in sorted(performance_rows, key=lambda x: -x['top1_acc']):
    bar = '#' * int(r['top1_acc'] * 30)
    print(f"  {r['model']:<18s}  {r['top1_acc']:.4f}  {bar}")

# Hien thi 1 so figures
show_figs = [
    'fig_all_gestures.png',
    'fig_confusion_cnn.png',
    'fig_confusion_comparison.png',
    'fig_f1_comparison.png',
    'fig_tsne_transformer.png',
    'fig_loss_acc_transformer.png',
]

for fname in show_figs:
    fp = fig_dir / fname
    if fp.exists():
        img = plt.imread(str(fp))
        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(fname, fontsize=10)
        plt.tight_layout()
        plt.show()

## 14. Lưu kết quả về Google Drive

In [ ]:
import shutil
from datetime import datetime

ts       = datetime.now().strftime('%Y%m%d_%H%M')
SAVE_DIR = DRIVE_ROOT / f'Do_an_results_{ts}'
SAVE_DIR.mkdir(exist_ok=True)

for src_rel, dst_name in [
    ('training',       'training'),
    ('demo/models',    'demo_models'),
    ('reports/figures','figures'),
]:
    src = PROJECT_DIR / src_rel
    dst = SAVE_DIR / dst_name
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Saved: {dst_name}/ -> Drive")
    else:
        print(f"Skip: {src_rel} (chua co)")

print(f"\nKet qua luu tai: MyDrive/Do_an_results_{ts}/")

## 15. (Tùy chọn) Tải zip trực tiếp về máy

In [ ]:
from google.colab import files

zip_path = f"/content/Do_an_results_{ts}"
shutil.make_archive(zip_path, 'zip', str(SAVE_DIR))
files.download(f"{zip_path}.zip")
print(f"Downloading: Do_an_results_{ts}.zip")